# CORE CAPSTONE — Public Services Coverage

Πλήρης λύση: audit → cleaning → κοινό CRS → spatial join → buffer/clip/overlay → QA → export.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

root = Path('data/core-capstone')
out = Path('output/core-capstone')
out.mkdir(parents=True, exist_ok=True)
study = gpd.read_file(root / 'study_area.gpkg')
zones = gpd.read_file(root / 'administrative_zones.gpkg')
roads = gpd.read_file(root / 'roads.gpkg')
services_raw = pd.read_csv(root / 'services.csv')
print(study.shape, zones.shape, roads.shape, services_raw.shape)


Το audit δείχνει ένα study polygon, τρεις ζώνες, τρεις roads και έντεκα raw records. Το CSV περιέχει σκόπιμα missing, duplicate, invalid και outside-area προβλήματα.

In [ ]:
print(services_raw.dtypes)
print(services_raw.isna().sum())
print(services_raw[services_raw.service_id.duplicated(keep=False)])
for frame in (study, zones, roads):
    print(frame.crs.to_epsg(), frame.geometry.geom_type.value_counts().to_dict(), frame.total_bounds)


## Cleaning

Μετατρέπουμε coordinates σε numeric ώστε το `not_available` να γίνει missing. Αφαιρούμε duplicate id, records χωρίς και τις δύο coordinates και, τέλος, σημεία έξω από το study area.

In [ ]:
services = services_raw.copy()
services['longitude'] = pd.to_numeric(services['longitude'], errors='coerce')
services['latitude'] = pd.to_numeric(services['latitude'], errors='coerce')
services = services.drop_duplicates('service_id', keep='first').dropna(subset=['longitude','latitude'])
services_gdf = gpd.GeoDataFrame(services, geometry=gpd.points_from_xy(services.longitude, services.latitude), crs='EPSG:4326')
study_wgs84 = study.to_crs('EPSG:4326')
services_gdf['inside_study'] = services_gdf.geometry.within(study_wgs84.union_all())
cleaned = services_gdf[services_gdf.inside_study].drop(columns='inside_study').copy()
print('raw:', len(services_raw), 'cleaned:', len(cleaned))


Το expected cleaned count είναι 7. Η πολιτική είναι συντηρητική: δεν επινοούμε συντεταγμένη για missing/invalid records και δεν αφήνουμε outside-area features να επηρεάσουν τη μελέτη.

In [ ]:
target_crs = 'EPSG:2100'
study_p = study.to_crs(target_crs)
zones_p = zones.to_crs(target_crs)
roads_p = roads.to_crs(target_crs)
services_p = cleaned.to_crs(target_crs)
for frame in (study_p, zones_p, roads_p, services_p):
    assert frame.crs.to_epsg() == 2100
    assert frame.geometry.notna().all() and frame.geometry.is_valid.all()


## Spatial join και exploration

Το join προσθέτει το zone_id χωρίς να κόψει τα σημεία. Με `how='left'` βλέπουμε και τυχόν unmatched services.

In [ ]:
services_with_zones = gpd.sjoin(services_p, zones_p[['zone_id','zone_name','geometry']], how='left', predicate='within')
assert len(services_with_zones) == 7
assert services_with_zones['zone_id'].notna().all()
print(services_with_zones.groupby(['zone_id','category']).size())


## Buffer, clip και overlay

Μετράμε απλή buffer-based κάλυψη 500 m γύρω από ενεργές Health/Emergency υπηρεσίες. Αυτό δεν είναι routing. Κάνουμε dissolve πριν από overlay ώστε επικαλυπτόμενες buffers να μη μετρηθούν δύο φορές.

In [ ]:
active_essential = services_p[(services_p.status == 'active') & (services_p.category.isin(['Health','Emergency']))].copy()
service_buffers = active_essential.copy()
service_buffers['geometry'] = service_buffers.geometry.buffer(500)
clipped_roads = gpd.clip(roads_p, study_p)
coverage_union = service_buffers[['geometry']].dissolve()
coverage_by_zone = gpd.overlay(zones_p, coverage_union, how='intersection')
coverage_by_zone['covered_m2'] = coverage_by_zone.geometry.area
zone_areas = zones_p[['zone_id','geometry']].copy()
zone_areas['zone_area_m2'] = zone_areas.geometry.area
coverage_by_zone = coverage_by_zone.merge(zone_areas[['zone_id','zone_area_m2']], on='zone_id')
coverage_by_zone['coverage_pct'] = 100 * coverage_by_zone.covered_m2 / coverage_by_zone.zone_area_m2


Το overlay επιστρέφει τις ζώνες που τέμνονται με την ενωμένη κάλυψη. Τα percentages είναι geometry-based, όχι population-weighted.

In [ ]:
assert cleaned.service_id.is_unique
assert len(service_buffers) == 3
assert (coverage_by_zone.covered_m2 > 0).all()
assert (coverage_by_zone.covered_m2 <= coverage_by_zone.zone_area_m2).all()
assert coverage_by_zone.zone_id.isin(zones_p.zone_id).all()
ax = zones_p.plot(facecolor='none', edgecolor='black', figsize=(9,7))
clipped_roads.plot(ax=ax, color='dimgray', linewidth=1, label='roads')
service_buffers.plot(ax=ax, color='orange', alpha=0.25, label='500 m coverage')
services_p.plot(ax=ax, color='navy', markersize=25, label='services')
ax.set_title('Public services coverage — QA map'); ax.legend(); plt.tight_layout()
plt.savefig(out / 'final_map.png', dpi=160)


## Export

Τα layers κρατούν διαφορετικό ρόλο: cleaned services για traceability, joined services για zone analysis, buffers για μέθοδο, coverage summary για απόφαση και clipped roads για context.

In [ ]:
gpkg = out / 'capstone_results.gpkg'
cleaned.to_file(gpkg, layer='cleaned_services', driver='GPKG')
services_with_zones.to_file(gpkg, layer='services_with_zones', driver='GPKG')
service_buffers.to_file(gpkg, layer='service_buffers', driver='GPKG')
coverage_by_zone.to_file(gpkg, layer='coverage_by_zone', driver='GPKG')
clipped_roads.to_file(gpkg, layer='clipped_roads', driver='GPKG')
summary = coverage_by_zone[['zone_id','zone_name','zone_area_m2','covered_m2','coverage_pct']].sort_values('zone_id')
summary.to_csv(out / 'summary.csv', index=False)
(out / 'short_findings.md').write_text('# Short findings\n\nΗ κάλυψη υπολογίστηκε με ενωμένες buffers 500 m γύρω από ενεργές Health/Emergency υπηρεσίες στο EPSG:2100. Τα αποτελέσματα είναι συνθετικά και δεν είναι routing ή population-weighted accessibility.\n', encoding='utf-8')
print('exported:', gpkg, summary)


## Τελικός έλεγχος

Ξαναδιάβασε τα output layers και επιβεβαίωσε ότι τα paths, CRS και row counts συμφωνούν με το audit. Αν αλλάξεις buffer distance ή business rule, επανάλαβε όλο το QA και ενημέρωσε τα findings.

In [ ]:
for layer in ['cleaned_services','services_with_zones','service_buffers','coverage_by_zone','clipped_roads']:
    check = gpd.read_file(gpkg, layer=layer)
    print(layer, len(check), check.crs.to_epsg(), check.geometry.geom_type.unique().tolist())
